# Stage 3 — Flow Matching before a frozen linear classifier

Stage 3 inserts an FM transformation between the frozen image-encoder feature and the Stage 1
linear probe:

$$z \;\xrightarrow{\;\text{FM}\;}\; \hat z \;\xrightarrow{\;\text{frozen linear classifier}\;}\; s$$

The classifier is trained first — it *is* the Stage 1 linear probe, loaded from disk, never
retrained — and then kept frozen. Only the velocity network is optimized. The direct baseline is
therefore the Stage 1 linear probe on the same dataset, encoder, K-shot subset and seed.

**What is inherited unchanged from Stages 1 and 2**

| From | What |
|---|---|
| Stage 1 | official splits, cached frozen features, the seeded K-shot subset selection, the trained linear head, and the feature transform that head expects |
| Stage 2 | the velocity network architecture, the Euler integrator `z_{k+1} = z_k + (1/T) v(z_k, k/T)`, the checkpoint-resume convention, and the deterministic per-batch `t` sampling |

**Two things are specific to Stage 3 and worth stating before any result is read.**

*The FM operates in the classifier's input space, not in raw feature space.* Stage 1 selected a
feature transform per run (`l2` or `standardize`) and trained the head on transformed features. So
`z` here is the transformed feature. Section 4 replays that transform and asserts that the reloaded
head reproduces Stage 1's saved test accuracy before anything is trained.

*The FM is initialized to the identity.* The velocity network's output layer is zeroed, so
`v(z, t) = 0` everywhere at initialization, the Euler rollout is exactly the identity map, and the
complete system starts at the linear probe rather than at a random displacement. This is what the
specification means by "initialize the FM close to identity"; here it is exact rather than close.
Section 6 asserts it.

That initialization has a consequence for model selection, and it is stated here rather than buried:
**epoch 0 — the untrained, identity FM — is evaluated and checkpointed like any other epoch.**
Validation accuracy of the selected checkpoint therefore can never fall below the linear probe's, so
validation ΔAcc ≥ 0 *by construction*. **Test** ΔAcc is not protected this way and is free to be
negative; it is the number to read.

## 1. Install dependencies

In [ ]:
%pip -q install scikit-learn pandas seaborn

## 2. Mount Drive and configure paths

In [ ]:
from pathlib import Path
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Flow-Matching')
else:
    PROJECT_ROOT = Path('/content/Flow-Matching')
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'fm_before_classifier'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Stage 3 outputs:', OUTPUT_ROOT)

## 3. Imports, hardware, and the Stage 3 plan

The plan is written into `stage1_config.json` as a `stage3` section on first run, following the
pattern Stage 2 used for its `flow_matching` section, so the configuration is versioned alongside
the Stage 1 plan rather than living only in this notebook.

**The choices the specification asks to fix, and why these:**

- **Datasets** — all three Stage 1 datasets. The specification asks for two; running the third costs
  little and keeps the Stage 3 table directly comparable to the Stage 1 and Stage 2 tables, which
  are three-dataset. Flowers-102 acts as a near-saturated control where no method has room to move.
- **Encoder** — DINOv2 ViT-S/14 on every dataset. Stage 2 measured that encoder choice dominates
  every other factor (+.14 to +.31 over ResNet-18); the representative encoder should be the one
  worth deploying.
- **K = 10**, the specification's suggested default, with Stage 1's three subset seeds {0, 1, 2}
  and `init_seed = 0` — exactly the repetition protocol Stage 1 used for its non-`full` settings.
- **T = 12**, fixed for training and inference in every Stage 3 method, matching the `T` used in
  Stage 2's headline figures.

Two hyperparameters deviate from Stage 2's FM config, both because of the identity initialization:
`early_stopping_patience` rises 25 → 40 and `lr_patience` 10 → 15. With a zeroed output layer only
that last layer receives gradient on the first step, so the first several epochs move very little;
Stage 2's patience would risk stopping a run before it left the identity.

In [ ]:
import copy, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'none')

DEFAULT_STAGE3_CONFIG = {
    'config_revision': 1,
    # --- the plan the specification asks to fix ---
    'datasets': ['dtd', 'aircraft', 'flowers102'],
    'encoder': 'dinov2_vits14',
    'shot': 10,
    'T': 12,
    'subset_seeds': [0, 1, 2],
    'init_seed': 0,
    # --- velocity network: identical to Stage 2, except the output layer is zeroed ---
    'hidden_dim': 512,
    'num_hidden_layers': 2,
    'zero_init_output': True,
    # --- optimization ---
    'batch_size': 64,
    'max_epochs': 200,
    'early_stopping_patience': 40,   # Stage 2 used 25; identity init needs a slower start
    'lr_patience': 15,               # Stage 2 used 10, same reason
    'min_delta': 1e-4,
    'checkpoint_interval': 10,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'resume_training': True,
    # --- Strategy 1 regularization (0 = the required, unregularized main result) ---
    'lambda_displacement': 0.0,
    'lambda_velocity': 0.0,
    # --- Strategy 2 classifier-guided targets ---
    'guidance_step_size': 0.1,       # as a fraction of the mean training-feature norm
    'guidance_num_steps': 3,
    'guidance_normalize': 'trust_region',
    'target_refresh_every': 1,       # epochs between target recomputations
    # --- optional extension: jointly fine-tuning the classifier ---
    'head_learning_rate': 1e-4,
    'unfreeze_epoch': 1,
}

config_paths = [
    PROJECT_ROOT / 'stage1_config.json',
    Path.cwd() / 'stage1_config.json',
    Path('/content/stage1_config.json'),
]
config_path = next((p for p in config_paths if p.exists()), None)
if config_path is None:
    raise FileNotFoundError('stage1_config.json not found. Run 01_linear_probe.ipynb first so the shared experiment plan exists.')
with config_path.open() as f:
    STAGE1 = json.load(f)
print('Loaded configuration from:', config_path)

if STAGE1.get('stage3', {}).get('config_revision', 0) < DEFAULT_STAGE3_CONFIG['config_revision']:
    STAGE1['stage3'] = copy.deepcopy(DEFAULT_STAGE3_CONFIG)
    config_path.write_text(json.dumps(STAGE1, indent=2))
    print('Added/updated stage3 config section, revision:', STAGE1['stage3']['config_revision'])

S3 = STAGE1['stage3']
DATASETS = S3['datasets']
ENCODER = S3['encoder']
SHOT = S3['shot']
T = S3['T']
SUBSET_SEEDS = S3['subset_seeds']
INIT_SEED = S3['init_seed']
CACHE_SCHEMA_VERSION = STAGE1['feature_cache']['schema_version']
CACHE_ROOT = PROJECT_ROOT / 'feature_cache' / f'v{CACHE_SCHEMA_VERSION}'
LINEAR_PROBE_ROOT = PROJECT_ROOT / 'outputs' / STAGE1['linear_probe']['output_subdir']

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(0)
sns.set_theme(style='whitegrid')
print(f'Datasets: {DATASETS} | encoder: {ENCODER} | K={SHOT} | T={T} | subset seeds {SUBSET_SEEDS}')
print('Linear probe root:', LINEAR_PROBE_ROOT)

## 4. Stage 1 artifacts: frozen features and the frozen linear classifier

Nothing is recomputed here. The frozen features come from Stage 1's cache and the classifier comes
from Stage 1's saved run directory.

`checkpoints/final.pt` is the artifact to load, not `best_linear_head.pt`. Stage 1 wrote `final.pt`
*after* `head.load_state_dict(best_state)`, so its `model_state_dict` holds the same
validation-selected weights as `best_linear_head.pt` — but `final.pt` is the only one that also
carries `feature_mean` / `feature_std`, which are required to rebuild the `standardize` transform
this head expects at its input.

The fidelity guard at the end of this section is the one that matters most in the whole notebook:
it replays the complete Stage 1 inference path (cache → subset → transform → head) on the test split
and requires the result to match Stage 1's saved `test_accuracy`. If the transform, the subset, or
the head were reconstructed even slightly wrongly, every Stage 3 ΔAcc would be measured against the
wrong baseline, and the effect would look like a Stage 3 result rather than a bug.

In [ ]:
def load_cached_features(dataset_name, encoder_name, split):
    cache = CACHE_ROOT / f'{dataset_name}__{encoder_name}__{split}.pt'
    if not cache.exists():
        raise FileNotFoundError(f'Missing Stage 1 feature cache: {cache}. Run 01_linear_probe.ipynb (Section 5) first.')
    return torch.load(cache, map_location='cpu', weights_only=False)

def balanced_indices(labels, k, seed):
    """Copied verbatim from 01_linear_probe.ipynb so Stage 3 re-derives the identical subset."""
    labels = np.asarray(labels); rng = np.random.default_rng(seed); chosen = []
    for c in np.unique(labels):
        idx = np.flatnonzero(labels == c)
        if len(idx) < k: raise ValueError(f'class {c} has {len(idx)} samples, fewer than K={k}')
        chosen.extend(rng.choice(idx, size=k, replace=False).tolist())
    return np.asarray(sorted(chosen))

def stage1_transform(x, mode, mean, std):
    """The Stage 1 feature transform, replayed. This defines the space the frozen head lives in,
    and therefore the space the FM layer operates in: z is a *transformed* feature, not a raw one."""
    if mode == 'l2':
        return F.normalize(x, dim=1)
    if mode == 'standardize':
        return (x - mean) / std
    if mode == 'none':
        return x
    raise ValueError(f'Unknown feature transform: {mode}')

def load_stage1_probe(dataset_name, encoder_name, shot, subset_seed, init_seed=0):
    """Load one frozen Stage 1 linear probe plus everything needed to reproduce its input space."""
    run_tag = f'subset_{subset_seed}__init_{init_seed}'
    run_dir = LINEAR_PROBE_ROOT / 'runs' / dataset_name / encoder_name / str(shot) / run_tag
    final_path, metrics_path = run_dir / 'checkpoints' / 'final.pt', run_dir / 'metrics.json'
    if not final_path.exists():
        raise FileNotFoundError(
            f'Missing Stage 1 linear probe: {final_path}. Run 01_linear_probe.ipynb first, and check '
            f'that its output_subdir is still "{STAGE1["linear_probe"]["output_subdir"]}".')
    if not metrics_path.exists():
        raise FileNotFoundError(f'Missing Stage 1 metrics: {metrics_path}.')
    checkpoint = torch.load(final_path, map_location='cpu', weights_only=False)
    metrics = json.loads(metrics_path.read_text())
    config, state = checkpoint['config'], checkpoint['model_state_dict']
    head = nn.Linear(state['weight'].shape[1], state['weight'].shape[0])
    head.load_state_dict(state)
    head.requires_grad_(False)          # frozen for the whole of Stage 3
    head.eval()
    return dict(head=head.to(DEVICE),
                feature_transform=config['feature_transform'],
                feature_mean=checkpoint['feature_mean'].float(),
                feature_std=checkpoint['feature_std'].float(),
                candidate=config.get('name'),
                stage1_test_accuracy=metrics['test_accuracy'],
                stage1_val_accuracy=metrics['best_val_accuracy'],
                run_dir=run_dir)

def build_stage3_inputs(bank, probe, shot, subset_seed):
    """Return the six tensors in the frozen classifier's own input space, on DEVICE.

    The transform statistics are recomputed from the re-derived K-shot subset and checked against
    the ones Stage 1 saved. If the subset were re-sampled differently, the standardize mean/std
    would drift and this fires before any training happens.
    """
    x_train, y_train = bank['train']['features'].float(), bank['train']['labels'].long()
    if shot != 'full':
        idx = torch.as_tensor(balanced_indices(y_train.numpy(), int(shot), subset_seed))
        x_train, y_train = x_train[idx], y_train[idx]
    x_val, y_val = bank['val']['features'].float(), bank['val']['labels'].long()
    x_test, y_test = bank['test']['features'].float(), bank['test']['labels'].long()

    mode = probe['feature_transform']
    if mode == 'standardize':
        mean = x_train.mean(0); std = x_train.std(0, unbiased=False).clamp_min(1e-5)
    else:
        mean, std = torch.zeros(x_train.shape[1]), torch.ones(x_train.shape[1])
    drift = max((mean - probe['feature_mean']).abs().max().item(),
                (std - probe['feature_std']).abs().max().item())
    if drift > 1e-4:
        raise RuntimeError(
            f'Feature-transform statistics disagree with Stage 1 by {drift:.2e}. The re-derived '
            f'K-shot subset is not the one Stage 1 trained on; Delta-accuracy would be meaningless.')

    return (stage1_transform(x_train, mode, mean, std).to(DEVICE), y_train.to(DEVICE),
            stage1_transform(x_val, mode, mean, std).to(DEVICE), y_val.to(DEVICE),
            stage1_transform(x_test, mode, mean, std).to(DEVICE), y_test.to(DEVICE))

feature_bank = {d: {split: load_cached_features(d, ENCODER, split) for split in ('train', 'val', 'test')}
                for d in DATASETS}
print('Loaded cached features for:', list(feature_bank))

### 4b. Fidelity guard — the reloaded probe must reproduce Stage 1's saved test accuracy

In [ ]:
@torch.no_grad()
def evaluate_system(net, head, z, y, T, chunk=4096):
    """Top-1 accuracy, mean cross-entropy, and predictions of head(rollout(z)) — the complete
    Stage 3 system. `net=None` evaluates the bare linear probe, i.e. the T-independent baseline."""
    if net is not None: net.eval()
    correct, total_loss, predictions = 0, 0.0, []
    for start in range(0, z.shape[0], chunk):
        zb, yb = z[start:start + chunk], y[start:start + chunk]
        z_hat = zb if net is None else euler_rollout(net, zb, T)
        logits = head(z_hat)
        total_loss += F.cross_entropy(logits, yb, reduction='sum').item()
        pred = logits.argmax(1)
        correct += (pred == yb).sum().item()
        predictions.append(pred.cpu())
    n = z.shape[0]
    return correct / n, total_loss / n, torch.cat(predictions)

STAGE3_DATA, PROBES, guard_rows = {}, {}, []
for dataset_name in DATASETS:
    for subset_seed in SUBSET_SEEDS:
        probe = load_stage1_probe(dataset_name, ENCODER, SHOT, subset_seed, INIT_SEED)
        data = build_stage3_inputs(feature_bank[dataset_name], probe, SHOT, subset_seed)
        STAGE3_DATA[(dataset_name, subset_seed)] = data
        PROBES[(dataset_name, subset_seed)] = probe
        z_test, y_test = data[4], data[5]
        replayed, replayed_ce, _ = evaluate_system(None, probe['head'], z_test, y_test, T)
        gap = abs(replayed - probe['stage1_test_accuracy'])
        guard_rows.append(dict(dataset=dataset_name, subset_seed=subset_seed,
                               transform=probe['feature_transform'], candidate=probe['candidate'],
                               stage1_test_accuracy=probe['stage1_test_accuracy'],
                               replayed_test_accuracy=replayed, abs_gap=gap,
                               n_train=data[0].shape[0], n_test=z_test.shape[0],
                               feature_dim=z_test.shape[1]))
        # Stage 1 stored accuracies as float32 means, so compare with a tolerance, never exactly.
        if gap > 1e-6:
            raise RuntimeError(
                f'Replayed Stage 1 probe disagrees with its saved test accuracy by {gap:.2e} '
                f'({dataset_name}, subset {subset_seed}). The Stage 3 input space is not Stage 1\'s.')
guard_df = pd.DataFrame(guard_rows)
display(guard_df)
print('Fidelity guard passed: every reloaded probe reproduces its Stage 1 test accuracy to <1e-6.')

## 5. The Euler integrator and the identity-initialized velocity network

Both are Stage 2's, unchanged in form. The only difference is that `zero_init_output` — which
existed in Stage 2 as a disabled pilot flag — is **required** here, because Stage 3 asks for an
FM initialized close to the identity.

With a zeroed output layer, `v(z, t) = 0` for every `z` and `t`, so each Euler step adds exactly
zero and `rollout(z) == z` bit for bit. The system at initialization is therefore not merely close
to the linear probe, it *is* the linear probe.

One consequence worth knowing when reading the training curves: at initialization the gradient
reaches only the output layer, because every earlier layer's gradient is multiplied by the zeroed
output weights. The hidden layers unblock after the first update. This is the standard behaviour of
zero-initialized residual branches, and it is why Section 3 raised the early-stopping patience.

In [ ]:
def euler_rollout(v_theta, z0, T, return_trajectory=False):
    """Stage 2's integrator, unchanged: z_{k+1} = z_k + (1/T) v(z_k, k/T)."""
    z = z0
    trajectory = [z0.detach().clone()] if return_trajectory else None
    for k in range(T):
        t = torch.full((z.shape[0],), k / T, device=z.device, dtype=z.dtype)
        z = z + (1.0 / T) * v_theta(z, t)
        if return_trajectory: trajectory.append(z.detach().clone())
    return (z, trajectory) if return_trajectory else z

class VelocityNet(nn.Module):
    """Stage 2's velocity network, identical in architecture: (feature_dim + 1) -> 512 -> 512 ->
    feature_dim with SiLU activations and the scalar flow time concatenated to the input."""
    def __init__(self, feature_dim, hidden_dim, num_hidden_layers, zero_init_output=True):
        super().__init__()
        layers = []
        in_dim = feature_dim + 1
        for _ in range(num_hidden_layers):
            layers += [nn.Linear(in_dim, hidden_dim), nn.SiLU()]
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, feature_dim))
        self.net = nn.Sequential(*layers)
        if zero_init_output:
            nn.init.zeros_(self.net[-1].weight)
            nn.init.zeros_(self.net[-1].bias)

    def forward(self, z, t):
        if t.dim() == 0: t = t.expand(z.shape[0])
        return self.net(torch.cat([z, t.view(-1, 1).to(z.dtype)], dim=1))

def make_velocity_net(feature_dim):
    return VelocityNet(feature_dim, S3['hidden_dim'], S3['num_hidden_layers'],
                       zero_init_output=S3['zero_init_output']).to(DEVICE)

### 5b. Identity guard — the untrained system must equal the linear probe exactly

In [ ]:
identity_rows = []
for dataset_name in DATASETS:
    for subset_seed in SUBSET_SEEDS:
        probe = PROBES[(dataset_name, subset_seed)]
        z_test, y_test = STAGE3_DATA[(dataset_name, subset_seed)][4:6]
        seed_everything(INIT_SEED)
        net0 = make_velocity_net(z_test.shape[1])
        with torch.no_grad():
            exact = torch.equal(euler_rollout(net0, z_test, T), z_test)
        acc0, _, _ = evaluate_system(net0, probe['head'], z_test, y_test, T)
        identity_rows.append(dict(dataset=dataset_name, subset_seed=subset_seed,
                                  rollout_is_identity=exact, system_at_init=acc0,
                                  linear_probe=probe['stage1_test_accuracy']))
        if not exact or abs(acc0 - probe['stage1_test_accuracy']) > 1e-6:
            raise RuntimeError(f'FM is not the identity at initialization for {dataset_name}/{subset_seed}.')

# The guard only has teeth if a default-initialized network would fail it.
seed_everything(0)
z_probe_test = STAGE3_DATA[(DATASETS[0], SUBSET_SEEDS[0])][4]
net_default = VelocityNet(z_probe_test.shape[1], S3['hidden_dim'], S3['num_hidden_layers'],
                          zero_init_output=False).to(DEVICE)
with torch.no_grad():
    moved = (euler_rollout(net_default, z_probe_test, T) - z_probe_test).norm(dim=1).mean().item()
display(pd.DataFrame(identity_rows))
print(f'Identity guard passed. A default-initialized network would instead displace features by '
      f'{moved:.4f} on average, so the guard is not vacuous.')

## 6. Strategy 1 — end-to-end rolled-out classification training

For each training feature `z`, run the complete `T`-step rollout to obtain `ẑ`, pass `ẑ` through the
frozen classifier, and minimize

$$\mathcal{L} = \mathrm{CE}(W\hat z + b,\; y) \;+\; \lambda_{\text{disp}}\,\lVert \hat z - z\rVert^2 \;+\; \frac{\lambda_{\text{vel}}}{T}\sum_k \lVert v(z_k, t_k)\rVert^2$$

with gradients flowing through all `T` sequential steps and updating only the FM parameters. The
head was frozen with `requires_grad_(False)` at load time, so it receives no gradient at all.

The two penalties are the ones the specification suggests, and they are off by default — the
required main result is the unregularized objective. Section 13 sweeps them.

**Why the penalties are worth having at all**, beyond the specification suggesting them: when
Stage 1 selected the `l2` transform, the head was only ever trained on unit-norm features. The
rollout is free to move `ẑ` off that sphere into regions where the head was never fit and its logits
are unconstrained — the FM can then win training accuracy by exploiting the classifier rather than by
improving the representation. Penalizing displacement is the direct guard against that, and the
sweep in Section 13 is where it gets measured rather than assumed.

In [ ]:
def strategy1_batch_loss(net, head, zb, yb, T, lambda_displacement, lambda_velocity):
    """CE on the endpoint of the full rollout, plus the two optional penalties. Gradients reach
    only the FM parameters — `head` was frozen at load."""
    z = zb
    velocity_sq = zb.new_zeros(())
    for k in range(T):
        t = torch.full((z.shape[0],), k / T, device=z.device, dtype=z.dtype)
        v = net(z, t)
        if lambda_velocity > 0: velocity_sq = velocity_sq + v.pow(2).sum(1).mean()
        z = z + (1.0 / T) * v
    loss_cls = F.cross_entropy(head(z), yb)
    loss = loss_cls
    if lambda_displacement > 0:
        loss = loss + lambda_displacement * (z - zb).pow(2).sum(1).mean()
    if lambda_velocity > 0:
        loss = loss + lambda_velocity * (velocity_sq / T)
    return loss, loss_cls.detach()

## 7. Strategy 2 — classifier-guided targets and standard FM training

The frozen classifier is used to *construct a target*, and the FM is then trained by ordinary
conditional flow matching between `z` and that target. Per the specification, for each training
feature `z`:

1. run `z` through the current FM to get `ẑ`;
2. pass `ẑ` through the frozen classifier and compute the classification loss;
3. take `n` gradient steps of that loss **with respect to `ẑ`** — in feature space, never in
   parameter space — to reach a nearby improved representation `ẑ'`;
4. treat `z` as the source and `ẑ'` as the target;
5. perform a standard FM update: `t ~ U(0,1)`, `z_t = (1-t)z + t ẑ'`, target velocity `ẑ' - z`,
   loss `‖v(z_t, t) - (ẑ' - z)‖²`;
6. recompute the targets as the FM changes — here every `target_refresh_every` epochs, over the
   whole training set.

The result is a bootstrapped scheme: each refresh moves the target a little further downhill in
classifier loss, and the FM chases it. Nothing pins the target to a fixed endpoint the way Stage 2's
class prototypes did, which is why the step constraint matters.

**Step size is expressed as a fraction of the mean training-feature norm, not as an absolute
distance.** Stage 1 chooses the feature transform per run, and an absolute step of 0.5 would be a
50% displacement under `l2` (‖z‖ = 1) but roughly 2.5% under `standardize` (‖z‖ ≈ √384). Making it
relative keeps one configured number comparable across datasets and transforms.

The three constraint modes the specification asks to experiment with are all implemented; the
default is `trust_region`. Under `none`, already-confident samples take naturally small steps
(their CE gradient is small) which is desirable, but raw gradient norms vary by orders of magnitude
across samples. Under `unit` every sample moves the same distance, including ones already classified
correctly. `trust_region` keeps the gradient's own scaling while capping the worst case. Section 14
measures whether that reasoning survives contact with the data.

In [ ]:
def classifier_guided_targets(net, head, z, y, T, step_size, n_steps, normalize, scale=1.0, chunk=4096):
    """Build the FM regression target z_hat' for every training feature.

    `step_size` is a fraction of `scale`, the mean training-feature norm. `normalize` is one of:
      'none'         - z_hat' = z_hat - eps * grad, raw gradient descent in feature space
      'unit'         - each step has length exactly eps, independent of the gradient magnitude
      'trust_region' - raw steps, with the cumulative displacement from z_hat clipped to eps
    """
    eps = step_size * scale
    if net is not None: net.eval()
    targets = []
    for start in range(0, z.shape[0], chunk):
        zb, yb = z[start:start + chunk], y[start:start + chunk]
        with torch.no_grad():
            z_hat = zb.clone() if net is None else euler_rollout(net, zb, T)
        anchor, current = z_hat, z_hat.clone()
        for _ in range(n_steps):
            current = current.detach().requires_grad_(True)
            loss = F.cross_entropy(head(current), yb, reduction='sum')
            grad, = torch.autograd.grad(loss, current)
            if normalize == 'unit':
                grad = grad / grad.norm(dim=1, keepdim=True).clamp_min(1e-12)
            current = current.detach() - eps * grad
            if normalize == 'trust_region':
                delta = current - anchor
                shrink = (eps / delta.norm(dim=1, keepdim=True).clamp_min(1e-12)).clamp(max=1.0)
                current = anchor + delta * shrink
        targets.append(current.detach())
    if net is not None: net.train()
    return torch.cat(targets)

def sample_flow_times(n, init_seed, epoch, batch_number):
    """Deterministic t ~ U(0,1), seeded per (init_seed, epoch, batch) exactly as in Stage 2, so an
    interrupted-and-resumed run samples the same t values as an uninterrupted one."""
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(init_seed * 1_000_000 + epoch * 10_000 + batch_number)
    return torch.rand(n, device=DEVICE, generator=generator)

def strategy2_batch_loss(net, zb, target_b, t):
    """Standard conditional-FM regression between source z and classifier-guided target z_hat'."""
    tv = t.view(-1, 1)
    zt = (1 - tv) * zb + tv * target_b
    return F.mse_loss(net(zt, t), target_b - zb)

## 8. One shared training loop for both strategies

Both strategies are checkpointed on the **same** quantity — validation top-1 accuracy of the
complete system `head(rollout(z_val))`. That is deliberate and it is what makes the comparison in
Section 10 fair: two objectives that optimize different losses are still selected by one rule, and
that rule is the same one Stage 1 used to select the linear probe. Selecting each method on its own
training loss instead would compare a well-tuned method against a badly-tuned one.

`epoch 0` is evaluated before any update. Because the FM is exactly the identity there, epoch 0 *is*
the linear probe, and seeding the checkpoint with it makes the selection rule "keep the FM only if
it helps on validation". The cost of that choice, restated: validation ΔAcc cannot be negative, so
only **test** ΔAcc carries information about whether Stage 3 helped. A run whose best epoch is 0
learned nothing usable and is reported as such rather than being quietly rounded away.

In [ ]:
def train_stage3(strategy, z_train, y_train, z_val, y_val, head, T, run_dir, run_config,
                 init_seed=0, hyper=None, joint_head=None, verbose=True):
    """Train one FM layer in front of the frozen classifier.

    `strategy` is 'e2e' (Section 6) or 'guided' (Section 7). `joint_head` is the optional extension
    of Section 15: pass a trainable copy of the head to unfreeze it and optimize it jointly.
    Returns (net, history, best_val_accuracy, best_epoch, runtime_seconds).
    """
    hyper = dict(hyper or {})
    seed_everything(init_seed)
    net = make_velocity_net(z_train.shape[1])

    param_groups = [dict(params=list(net.parameters()), lr=S3['learning_rate'])]
    unfreeze_epoch = hyper.get('unfreeze_epoch', S3['unfreeze_epoch'])
    if joint_head is not None:
        joint_head.requires_grad_(True)
        param_groups.append(dict(params=list(joint_head.parameters()),
                                 lr=hyper.get('head_learning_rate', S3['head_learning_rate'])))
    optimizer = torch.optim.AdamW(param_groups, weight_decay=S3['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                                                           patience=S3['lr_patience'])
    active_head = head if joint_head is None else joint_head

    lambda_disp = hyper.get('lambda_displacement', S3['lambda_displacement'])
    lambda_vel = hyper.get('lambda_velocity', S3['lambda_velocity'])
    guide_step = hyper.get('guidance_step_size', S3['guidance_step_size'])
    guide_nsteps = hyper.get('guidance_num_steps', S3['guidance_num_steps'])
    guide_norm = hyper.get('guidance_normalize', S3['guidance_normalize'])
    refresh_every = max(1, hyper.get('target_refresh_every', S3['target_refresh_every']))

    checkpoint_dir = run_dir / 'checkpoints'; checkpoint_dir.mkdir(parents=True, exist_ok=True)
    latest_path, best_path = checkpoint_dir / 'latest.pt', checkpoint_dir / 'best.pt'
    (run_dir / 'config.json').write_text(json.dumps(run_config, indent=2))

    best_acc, best_state, best_head_state, best_epoch = -1.0, None, None, None
    history, start_epoch, stale_epochs = [], 1, 0

    if S3['resume_training'] and latest_path.exists():
        checkpoint = torch.load(latest_path, map_location=DEVICE, weights_only=False)
        if checkpoint.get('config') == run_config:
            net.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if checkpoint.get('scheduler_state_dict') is not None:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if joint_head is not None and checkpoint.get('head_state_dict') is not None:
                joint_head.load_state_dict(checkpoint['head_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_acc, best_epoch = checkpoint['best_val_accuracy'], checkpoint.get('best_epoch')
            best_state, best_head_state = checkpoint['best_model_state_dict'], checkpoint.get('best_head_state_dict')
            history, stale_epochs = checkpoint['history'], checkpoint.get('stale_epochs', 0)
            if verbose: print(f'Resuming {run_dir.name} from epoch {start_epoch}')
        elif verbose:
            print(f'Checkpoint configuration changed for {run_dir.name}; starting a fresh run.')

    def save_checkpoint(path, epoch):
        torch.save(dict(epoch=epoch, model_state_dict=net.state_dict(),
                        optimizer_state_dict=optimizer.state_dict(),
                        scheduler_state_dict=scheduler.state_dict(),
                        best_model_state_dict=best_state, best_head_state_dict=best_head_state,
                        head_state_dict=joint_head.state_dict() if joint_head is not None else None,
                        best_val_accuracy=best_acc, best_epoch=best_epoch,
                        stale_epochs=stale_epochs, history=history, config=run_config), path)

    started = time.perf_counter()
    n_train = z_train.shape[0]
    targets = None
    guide_scale = z_train.norm(dim=1).mean().item()   # guidance steps are relative to feature scale

    # Epoch 0: the identity-initialized system, evaluated before any update. It *is* the Stage 1
    # linear probe, so seeding the checkpoint with it makes "keep the FM only if it helps" the rule.
    if start_epoch == 1 and not history:
        init_acc, init_loss, _ = evaluate_system(net, active_head, z_val, y_val, T)
        best_acc, best_epoch = init_acc, 0
        best_state = copy.deepcopy(net.state_dict())
        if joint_head is not None: best_head_state = copy.deepcopy(joint_head.state_dict())
        history.append(dict(epoch=0, train_loss=float('nan'), train_cls_loss=float('nan'),
                            val_accuracy=init_acc, val_loss=init_loss,
                            learning_rate=optimizer.param_groups[0]['lr']))
        save_checkpoint(best_path, 0)

    for epoch in range(start_epoch, S3['max_epochs'] + 1):
        if joint_head is not None:
            joint_head.requires_grad_(epoch >= unfreeze_epoch)
        if strategy == 'guided' and (targets is None or (epoch - 1) % refresh_every == 0):
            targets = classifier_guided_targets(net, active_head, z_train, y_train, T,
                                                guide_step, guide_nsteps, guide_norm, guide_scale)
        generator = torch.Generator().manual_seed(init_seed * 100000 + epoch)
        perm = torch.randperm(n_train, generator=generator)
        net.train(); train_total, cls_total = 0.0, 0.0
        for batch_number, start in enumerate(range(0, n_train, S3['batch_size'])):
            batch_idx = perm[start:start + S3['batch_size']]
            zb, yb = z_train[batch_idx], y_train[batch_idx]
            optimizer.zero_grad(set_to_none=True)
            if strategy == 'e2e':
                loss, loss_cls = strategy1_batch_loss(net, active_head, zb, yb, T, lambda_disp, lambda_vel)
            elif strategy == 'guided':
                t = sample_flow_times(zb.shape[0], init_seed, epoch, batch_number)
                loss = strategy2_batch_loss(net, zb, targets[batch_idx], t)
                loss_cls = loss.detach()
            else:
                raise ValueError(f'Unknown strategy: {strategy}')
            loss.backward(); optimizer.step()
            train_total += loss.item() * len(batch_idx)
            cls_total += loss_cls.item() * len(batch_idx)
        val_acc, val_loss, _ = evaluate_system(net, active_head, z_val, y_val, T)
        net.train()
        history.append(dict(epoch=epoch, train_loss=train_total / n_train,
                            train_cls_loss=cls_total / n_train, val_accuracy=val_acc,
                            val_loss=val_loss, learning_rate=optimizer.param_groups[0]['lr']))
        if val_acc > best_acc + S3['min_delta']:
            best_acc, best_epoch = val_acc, epoch
            best_state = copy.deepcopy(net.state_dict())
            if joint_head is not None: best_head_state = copy.deepcopy(joint_head.state_dict())
            stale_epochs = 0
            save_checkpoint(best_path, epoch)
        else:
            stale_epochs += 1
        scheduler.step(val_acc)
        if epoch % S3['checkpoint_interval'] == 0: save_checkpoint(latest_path, epoch)
        if stale_epochs >= S3['early_stopping_patience']:
            save_checkpoint(latest_path, epoch)
            if verbose: print(f'  early stopping at epoch {epoch}')
            break
    if history: save_checkpoint(latest_path, history[-1]['epoch'])
    if best_state is None: best_state = copy.deepcopy(net.state_dict())
    runtime_seconds = time.perf_counter() - started
    net.load_state_dict(best_state); net.eval()
    if joint_head is not None and best_head_state is not None:
        joint_head.load_state_dict(best_head_state); joint_head.eval()
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    return net, history, best_acc, best_epoch, runtime_seconds

## 9. Run the main grid

3 datasets × 1 encoder × K=10 × 3 subset seeds × 2 strategies = **18 trained FM layers**. The Stage 1
linear probe baseline is *loaded*, never retrained, so it contributes 9 more result rows at zero
training cost.

Completed runs are skipped by checking for saved outputs on Drive, matching the Stage 1 and Stage 2
convention. Set `FORCE_RETRAIN = True` to ignore them.

In [ ]:
FORCE_RETRAIN = False   # set True to ignore saved results on Drive and rerun the whole grid

STRATEGIES = [('e2e', 'end-to-end rollout'), ('guided', 'classifier-guided')]

def run_directory(tag, dataset_name, subset_seed, root=None):
    return (root or OUTPUT_ROOT) / 'runs' / dataset_name / ENCODER / str(SHOT) / tag / f'subset_{subset_seed}'

def load_completed(run_dir, run_config):
    metrics_path, history_path, pred_path = (run_dir / 'metrics.json', run_dir / 'history.csv',
                                             run_dir / 'test_predictions.npy')
    config_path = run_dir / 'config.json'
    if not all(p.exists() for p in (metrics_path, history_path, pred_path, config_path)):
        return None
    if json.loads(config_path.read_text()) != run_config:
        return None
    return (json.loads(metrics_path.read_text()),
            pd.read_csv(history_path).to_dict('records'),
            torch.from_numpy(np.load(pred_path)))

def run_one(strategy, dataset_name, subset_seed, tag=None, hyper=None, root=None, joint=False,
            verbose=True):
    """Train (or reload) one Stage 3 cell and evaluate it on the untouched test split."""
    hyper = dict(hyper or {})
    tag = tag or strategy
    probe = PROBES[(dataset_name, subset_seed)]
    z_tr, y_tr, z_va, y_va, z_te, y_te = STAGE3_DATA[(dataset_name, subset_seed)]
    run_dir = run_directory(tag, dataset_name, subset_seed, root); run_dir.mkdir(parents=True, exist_ok=True)
    run_config = dict(stage='stage3', strategy=strategy, tag=tag, dataset=dataset_name,
                      encoder=ENCODER, shot=str(SHOT), subset_seed=subset_seed, init_seed=INIT_SEED,
                      T=T, feature_dim=int(z_tr.shape[1]), num_classes=int(probe['head'].out_features),
                      feature_transform=probe['feature_transform'], joint_finetune=joint,
                      hidden_dim=S3['hidden_dim'], num_hidden_layers=S3['num_hidden_layers'],
                      learning_rate=S3['learning_rate'], weight_decay=S3['weight_decay'],
                      zero_init_output=S3['zero_init_output'], hyper=hyper)

    cached = None if FORCE_RETRAIN else load_completed(run_dir, run_config)
    if cached is not None:
        metrics, history, test_pred = cached
        if verbose: print(f'  loaded  {tag:24s} {dataset_name:11s} subset {subset_seed}  '
                          f"test {metrics['test_accuracy']:.4f}")
        return metrics, history, test_pred

    joint_head = copy.deepcopy(probe['head']) if joint else None
    net, history, best_val, best_epoch, runtime = train_stage3(
        strategy, z_tr, y_tr, z_va, y_va, probe['head'], T, run_dir, run_config,
        init_seed=INIT_SEED, hyper=hyper, joint_head=joint_head, verbose=verbose)
    eval_head_module = joint_head if joint_head is not None else probe['head']
    test_acc, test_ce, test_pred = evaluate_system(net, eval_head_module, z_te, y_te, T)
    with torch.no_grad():
        z_hat = euler_rollout(net, z_te, T)
        displacement = (z_hat - z_te).norm(dim=1).mean().item()
        relative_displacement = displacement / z_te.norm(dim=1).mean().item()
    metrics = dict(dataset=dataset_name, encoder=ENCODER, shot=str(SHOT), subset_seed=subset_seed,
                   strategy=strategy, tag=tag, T=T, joint_finetune=joint,
                   baseline_test_accuracy=probe['stage1_test_accuracy'],
                   baseline_val_accuracy=probe['stage1_val_accuracy'],
                   best_val_accuracy=best_val, best_epoch=best_epoch,
                   epochs_trained=len(history) - 1, test_accuracy=test_acc, test_loss=test_ce,
                   delta_accuracy=test_acc - probe['stage1_test_accuracy'],
                   mean_displacement=displacement, relative_displacement=relative_displacement,
                   runtime_seconds=runtime, feature_transform=probe['feature_transform'])
    (run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    np.save(run_dir / 'test_predictions.npy', test_pred.numpy())
    if joint_head is not None:
        torch.save(joint_head.state_dict(), run_dir / 'finetuned_head.pt')
    if verbose:
        print(f'  trained {tag:24s} {dataset_name:11s} subset {subset_seed}  '
              f"test {test_acc:.4f} ({metrics['delta_accuracy']:+.4f})  best@{best_epoch}  {runtime:.0f}s")
    return metrics, history, test_pred

main_rows, main_histories, main_predictions = [], {}, {}
for dataset_name in DATASETS:
    print(f'\n=== {dataset_name} | {ENCODER} | K={SHOT} | T={T} ===')
    for subset_seed in SUBSET_SEEDS:
        probe = PROBES[(dataset_name, subset_seed)]
        z_te, y_te = STAGE3_DATA[(dataset_name, subset_seed)][4:6]
        _, _, probe_pred = evaluate_system(None, probe['head'], z_te, y_te, T)
        main_rows.append(dict(dataset=dataset_name, encoder=ENCODER, shot=str(SHOT),
                              subset_seed=subset_seed, strategy='linear_probe', tag='linear_probe',
                              T=None, joint_finetune=False,
                              baseline_test_accuracy=probe['stage1_test_accuracy'],
                              best_val_accuracy=probe['stage1_val_accuracy'], best_epoch=None,
                              epochs_trained=0, test_accuracy=probe['stage1_test_accuracy'],
                              delta_accuracy=0.0, mean_displacement=0.0, relative_displacement=0.0,
                              runtime_seconds=0.0, feature_transform=probe['feature_transform']))
        main_predictions[('linear_probe', dataset_name, subset_seed)] = probe_pred
        for strategy, _label in STRATEGIES:
            metrics, history, pred = run_one(strategy, dataset_name, subset_seed)
            main_rows.append(metrics)
            main_histories[(strategy, dataset_name, subset_seed)] = history
            main_predictions[(strategy, dataset_name, subset_seed)] = pred

results_df = pd.DataFrame(main_rows)
results_df.to_csv(OUTPUT_ROOT / 'run_metrics.csv', index=False)
display(results_df)

## 10. Classification results

Top-1 accuracy on the complete official test split, mean ± std over the three Stage 1 subset seeds,
with ΔAcc against the matching linear probe — same dataset, same encoder, same K-shot subset, same
seed. The comparison is paired at the seed level, so the ΔAcc column is a mean of per-seed
differences, not a difference of means computed over different data.

In [ ]:
METHOD_ORDER = ['linear_probe', 'e2e', 'guided']
METHOD_LABEL = {'linear_probe': 'Stage 1 linear probe',
                'e2e': 'Stage 3 end-to-end rollout',
                'guided': 'Stage 3 classifier-guided'}

summary = (results_df.groupby(['dataset', 'strategy'], as_index=False)
           .agg(mean_accuracy=('test_accuracy', 'mean'), std_accuracy=('test_accuracy', 'std'),
                mean_delta=('delta_accuracy', 'mean'), std_delta=('delta_accuracy', 'std'),
                mean_best_epoch=('best_epoch', 'mean'), n=('test_accuracy', 'size')))
summary['strategy'] = pd.Categorical(summary['strategy'], METHOD_ORDER, ordered=True)
summary = summary.sort_values(['dataset', 'strategy'])
summary.to_csv(OUTPUT_ROOT / 'accuracy_summary.csv', index=False)
display(summary)

table = summary.pivot(index='dataset', columns='strategy', values='mean_accuracy')
print('\nTop-1 test accuracy (mean over 3 subset seeds), with ΔAcc vs the linear probe:\n')
header = f"{'dataset':<12}" + ''.join(f'{METHOD_LABEL[m]:>30}' for m in METHOD_ORDER)
print(header); print('-' * len(header))
for dataset_name in DATASETS:
    row = f'{dataset_name:<12}'
    for method in METHOD_ORDER:
        part = summary[(summary.dataset == dataset_name) & (summary.strategy == method)]
        if part.empty: row += f"{'-':>30}"; continue
        acc, sd, delta = part.iloc[0][['mean_accuracy', 'std_accuracy', 'mean_delta']]
        cell = f'{acc:.4f}±{sd:.4f}' if method == 'linear_probe' else f'{acc:.4f} ({delta:+.4f})'
        row += f'{cell:>30}'
    print(row)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_df = summary[summary.strategy != 'linear_probe']
sns.barplot(data=summary, x='dataset', y='mean_accuracy', hue='strategy', ax=axes[0],
            order=DATASETS, hue_order=METHOD_ORDER)
axes[0].set_title(f'Top-1 test accuracy — {ENCODER}, K={SHOT}, T={T}')
axes[0].set_ylabel('accuracy'); axes[0].legend(title='', fontsize=8)
sns.barplot(data=plot_df, x='dataset', y='mean_delta', hue='strategy', ax=axes[1],
            order=DATASETS, hue_order=METHOD_ORDER[1:])
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('ΔAcc against the Stage 1 linear probe')
axes[1].set_ylabel('Δ accuracy'); axes[1].legend(title='', fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_ROOT / 'accuracy_and_delta.png', dpi=150); plt.show()

identity_kept = results_df[(results_df.strategy != 'linear_probe') & (results_df.best_epoch == 0)]
print(f'\nRuns whose selected checkpoint was epoch 0 (the FM stayed at the identity): '
      f'{len(identity_kept)} of {len(results_df[results_df.strategy != "linear_probe"])}')
if len(identity_kept):
    display(identity_kept[['dataset', 'strategy', 'subset_seed', 'best_val_accuracy', 'test_accuracy']])

## 11. Training and validation behaviour

Both curves come from the same runs the table above reports. Two reading notes:

- **Epoch 0 is the identity FM**, so the validation-accuracy curve starts exactly at the linear
  probe's validation accuracy. Its train loss is `NaN` by construction — no update has happened —
  and is omitted from the loss panel.
- The two strategies minimize **different quantities**. Strategy 1's training loss is the
  classification cross-entropy on the rollout endpoint; Strategy 2's is a velocity regression MSE
  against a target that itself moves every `target_refresh_every` epochs. They are plotted on
  separate axes and must not be compared by magnitude. A Strategy 2 loss that *rises* is not
  divergence — it usually means the target moved further away after a refresh.

In [ ]:
CURVE_SEED = SUBSET_SEEDS[0]
fig, axes = plt.subplots(3, len(DATASETS), figsize=(6 * len(DATASETS), 12), squeeze=False)
for col, dataset_name in enumerate(DATASETS):
    probe = PROBES[(dataset_name, CURVE_SEED)]
    for strategy, label in STRATEGIES:
        history = pd.DataFrame(main_histories[(strategy, dataset_name, CURVE_SEED)])
        trained = history[history.epoch > 0]
        ax = axes[0][col] if strategy == 'e2e' else axes[1][col]
        ax.plot(trained.epoch, trained.train_loss, label=f'{label} — train loss')
        ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.set_ylabel('training loss')
        ax.set_title(f'{dataset_name} — {label}\n(training objective, not comparable across rows)')
        ax.legend(fontsize=8)
        axes[2][col].plot(history.epoch, history.val_accuracy, label=f'{label}')
    axes[2][col].axhline(probe['stage1_val_accuracy'], color='black', linestyle='--', linewidth=1,
                         label='Stage 1 linear probe')
    axes[2][col].set_xlabel('epoch'); axes[2][col].set_ylabel('validation accuracy')
    axes[2][col].set_title(f'{dataset_name} — validation accuracy of the complete system')
    axes[2][col].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_ROOT / 'training_curves.png', dpi=150); plt.show()

behaviour = (results_df[results_df.strategy != 'linear_probe']
             .groupby('strategy', as_index=False)
             .agg(mean_epochs=('epochs_trained', 'mean'), mean_best_epoch=('best_epoch', 'mean'),
                  mean_runtime_s=('runtime_seconds', 'mean'),
                  mean_relative_displacement=('relative_displacement', 'mean')))
display(behaviour)
print('mean_relative_displacement is ||z_hat - z|| / ||z||, averaged over the test split: how far '
      'the FM actually moves the representation it was initialized to leave untouched.')

## 12. Feature-space visualization

For a readable subset of classes, the original features `z` and the transported features `ẑ` under
both Stage 3 methods, on the **same test examples with the same class colors** across all three
panels.

Following the Stage 2 convention, the 2D embedding is fit **jointly** over all three feature sets
being compared, and the three panels then **share axis limits**. A joint fit that is autoscaled per
panel would rescale each view independently and hide exactly the movement the figure exists to show.

In [ ]:
N_VIS_CLASSES = 10
VIS_PER_CLASS = 30
VIS_SEED = SUBSET_SEEDS[0]

def visualization_sample(dataset_name, subset_seed):
    z_te, y_te = STAGE3_DATA[(dataset_name, subset_seed)][4:6]
    y_np = y_te.cpu().numpy()
    classes = np.unique(y_np)[:N_VIS_CLASSES]
    rng = np.random.default_rng(0); chosen = []
    for c in classes:
        idx = np.flatnonzero(y_np == c)
        chosen.extend(rng.choice(idx, size=min(VIS_PER_CLASS, len(idx)), replace=False).tolist())
    chosen = np.asarray(sorted(chosen))
    return z_te[chosen], y_np[chosen], classes

def transported_views(dataset_name, subset_seed, z_subset):
    views = {'original': z_subset.cpu().numpy()}
    for strategy, label in STRATEGIES:
        run_dir = run_directory(strategy, dataset_name, subset_seed)
        checkpoint = torch.load(run_dir / 'checkpoints' / 'best.pt', map_location=DEVICE, weights_only=False)
        net = make_velocity_net(z_subset.shape[1])
        net.load_state_dict(checkpoint['best_model_state_dict'] or checkpoint['model_state_dict'])
        net.eval()
        with torch.no_grad():
            views[label] = euler_rollout(net, z_subset, T).cpu().numpy()
    return views

def joint_projection(views, method='pca'):
    stacked = np.concatenate(list(views.values()), axis=0)
    if method == 'pca':
        embedded = PCA(n_components=2, random_state=0).fit_transform(stacked)
    else:
        perplexity = min(30, max(5, stacked.shape[0] // 4 - 1))
        embedded = TSNE(n_components=2, random_state=0, init='pca',
                        perplexity=perplexity).fit_transform(stacked)
    n = next(iter(views.values())).shape[0]
    return {name: embedded[i * n:(i + 1) * n] for i, name in enumerate(views)}

def plot_joint_views(method):
    fig, axes = plt.subplots(len(DATASETS), 3, figsize=(16, 5 * len(DATASETS)), squeeze=False)
    for row, dataset_name in enumerate(DATASETS):
        z_subset, y_subset, classes = visualization_sample(dataset_name, VIS_SEED)
        views = transported_views(dataset_name, VIS_SEED, z_subset)
        embedded = joint_projection(views, method)
        all_points = np.concatenate(list(embedded.values()), axis=0)
        xlim = (all_points[:, 0].min() - 1, all_points[:, 0].max() + 1)
        ylim = (all_points[:, 1].min() - 1, all_points[:, 1].max() + 1)
        for col, (name, points) in enumerate(embedded.items()):
            ax = axes[row][col]
            for i, c in enumerate(classes):
                mask = y_subset == c
                ax.scatter(points[mask, 0], points[mask, 1], s=10, alpha=0.75,
                           color=plt.cm.tab10(i % 10), label=f'class {c}' if col == 0 else None)
            ax.set_xlim(*xlim); ax.set_ylim(*ylim)
            ax.set_title(f'{dataset_name} — {name}')
        axes[row][0].legend(fontsize=6, ncol=2, loc='best')
    plt.suptitle(f'Joint {method.upper()} over all three views — same test examples, same colors, '
                 f'shared axes', y=1.002)
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f'feature_space_{method}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_joint_views('pca')

### 12b. The same comparison under a joint t-SNE

In [ ]:
plot_joint_views('tsne')

## 13. Is the difference real? Paired confidence intervals and McNemar

The three-seed mean in Section 10 has no error bar worth trusting on its own. Two paired tests are
run per (dataset, method), both against the linear probe **on the same test split and the same
seed**:

- a percentile bootstrap over test *examples*, giving a 95% CI for ΔAcc;
- an exact McNemar test on the discordant pairs, which is the appropriate test for two classifiers
  evaluated on one sample.

These measure test-set uncertainty, not seed uncertainty. A ΔAcc whose CI spans zero should be
reported as "no measurable difference", not as a small win.

In [ ]:
from scipy import stats

def paired_bootstrap_ci(correct_a, correct_b, n_resamples=10000, seed=0):
    a = np.asarray(correct_a, dtype=float); b = np.asarray(correct_b, dtype=float)
    rng = np.random.default_rng(seed); n = len(a)
    idx = rng.integers(0, n, size=(n_resamples, n))
    diffs = a[idx].mean(1) - b[idx].mean(1)
    return float(a.mean() - b.mean()), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

def mcnemar_exact(correct_a, correct_b):
    a = np.asarray(correct_a, dtype=bool); b = np.asarray(correct_b, dtype=bool)
    b10 = int((a & ~b).sum())      # Stage 3 right, probe wrong
    b01 = int((~a & b).sum())      # probe right, Stage 3 wrong
    n = b01 + b10
    p = 1.0 if n == 0 else float(stats.binomtest(b10, n, 0.5).pvalue)
    return b01, b10, p

sig_rows = []
for dataset_name in DATASETS:
    for subset_seed in SUBSET_SEEDS:
        y_te = STAGE3_DATA[(dataset_name, subset_seed)][5].cpu()
        probe_correct = (main_predictions[('linear_probe', dataset_name, subset_seed)] == y_te).numpy()
        for strategy, label in STRATEGIES:
            fm_correct = (main_predictions[(strategy, dataset_name, subset_seed)] == y_te).numpy()
            delta, lo, hi = paired_bootstrap_ci(fm_correct, probe_correct)
            b01, b10, p = mcnemar_exact(fm_correct, probe_correct)
            sig_rows.append(dict(dataset=dataset_name, strategy=strategy, subset_seed=subset_seed,
                                 delta=delta, ci_low=lo, ci_high=hi,
                                 probe_only_correct=b01, fm_only_correct=b10, mcnemar_p=p,
                                 significant=bool(p < 0.05), ci_excludes_zero=bool(lo > 0 or hi < 0)))
significance_df = pd.DataFrame(sig_rows)
significance_df.to_csv(OUTPUT_ROOT / 'paired_significance.csv', index=False)
display(significance_df)

n_cells = len(significance_df)
print(f'\nOf {n_cells} (dataset, method, seed) cells:')
print(f'  {int((significance_df.delta > 0).sum())} favour Stage 3, '
      f'{int((significance_df.delta < 0).sum())} favour the linear probe, '
      f'{int((significance_df.delta == 0).sum())} are exactly tied')
print(f'  {int(significance_df.ci_excludes_zero.sum())} have a bootstrap CI excluding zero')
print(f'  {int(significance_df.significant.sum())} reach McNemar p < 0.05')

## 14. Variant comparison A — does regularizing the displacement help Strategy 1?

The specification invites experimenting with penalties on the displacement `‖ẑ - z‖` and on the
predicted velocity magnitude. This sweep runs on **subset seed 0 only** — it is a variant comparison,
not a replacement for the three-seed main result — and reports both accuracy and how far the FM
actually moved the representation.

The hypothesis being tested is the one stated in Section 6: an unregularized rollout can push `ẑ`
off the manifold the frozen head was fit on and win training accuracy by exploiting the classifier.
If that is happening, the unregularized runs should show large `relative_displacement` together with
a train/test gap that the penalized runs do not have.

In [ ]:
RUN_REGULARIZATION_SWEEP = True
REG_SWEEP = [
    ('e2e__reg_none',      dict()),
    ('e2e__disp_1e-2',     dict(lambda_displacement=1e-2)),
    ('e2e__disp_1e-1',     dict(lambda_displacement=1e-1)),
    ('e2e__vel_1e-3',      dict(lambda_velocity=1e-3)),
]
SWEEP_SEED = SUBSET_SEEDS[0]

if RUN_REGULARIZATION_SWEEP:
    sweep_rows = []
    for dataset_name in DATASETS:
        print(f'\n=== regularization sweep | {dataset_name} | subset {SWEEP_SEED} ===')
        for tag, hyper in REG_SWEEP:
            metrics, _, _ = run_one('e2e', dataset_name, SWEEP_SEED, tag=tag, hyper=hyper,
                                    root=OUTPUT_ROOT / 'sweeps')
            sweep_rows.append(dict(metrics, variant=tag,
                                   lambda_displacement=hyper.get('lambda_displacement', 0.0),
                                   lambda_velocity=hyper.get('lambda_velocity', 0.0)))
    reg_df = pd.DataFrame(sweep_rows)
    reg_df.to_csv(OUTPUT_ROOT / 'sweep_regularization.csv', index=False)
    display(reg_df[['dataset', 'variant', 'test_accuracy', 'delta_accuracy',
                    'relative_displacement', 'best_epoch', 'best_val_accuracy']])
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    sns.barplot(data=reg_df, x='dataset', y='delta_accuracy', hue='variant', ax=axes[0], order=DATASETS)
    axes[0].axhline(0, color='black', linewidth=1); axes[0].set_title('ΔAcc vs the linear probe')
    axes[0].legend(fontsize=7)
    sns.barplot(data=reg_df, x='dataset', y='relative_displacement', hue='variant', ax=axes[1], order=DATASETS)
    axes[1].set_title('mean ||ẑ - z|| / ||z|| on the test split'); axes[1].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(OUTPUT_ROOT / 'sweep_regularization.png', dpi=150); plt.show()
else:
    print('Regularization sweep disabled (RUN_REGULARIZATION_SWEEP = False).')

## 15. Variant comparison B — the Strategy 2 guidance knobs

The specification names four things to experiment with: the feature-space step size, the number of
target-improvement steps, how often targets are recomputed, and whether the target updates are
normalized or otherwise constrained. This sweep varies each in turn from the default, again on
subset seed 0 only.

In [ ]:
RUN_GUIDANCE_SWEEP = True
GUIDANCE_SWEEP = [
    ('guided__default',        dict()),
    ('guided__step_0.02',      dict(guidance_step_size=0.02)),
    ('guided__step_0.30',      dict(guidance_step_size=0.30)),
    ('guided__steps_1',        dict(guidance_num_steps=1)),
    ('guided__steps_10',       dict(guidance_num_steps=10)),
    ('guided__norm_none',      dict(guidance_normalize='none')),
    ('guided__norm_unit',      dict(guidance_normalize='unit')),
    ('guided__refresh_10',     dict(target_refresh_every=10)),
]

if RUN_GUIDANCE_SWEEP:
    guide_rows = []
    for dataset_name in DATASETS:
        print(f'\n=== guidance sweep | {dataset_name} | subset {SWEEP_SEED} ===')
        for tag, hyper in GUIDANCE_SWEEP:
            metrics, _, _ = run_one('guided', dataset_name, SWEEP_SEED, tag=tag, hyper=hyper,
                                    root=OUTPUT_ROOT / 'sweeps')
            guide_rows.append(dict(metrics, variant=tag,
                                   step_size=hyper.get('guidance_step_size', S3['guidance_step_size']),
                                   num_steps=hyper.get('guidance_num_steps', S3['guidance_num_steps']),
                                   normalize=hyper.get('guidance_normalize', S3['guidance_normalize']),
                                   refresh_every=hyper.get('target_refresh_every', S3['target_refresh_every'])))
    guidance_df = pd.DataFrame(guide_rows)
    guidance_df.to_csv(OUTPUT_ROOT / 'sweep_guidance.csv', index=False)
    display(guidance_df[['dataset', 'variant', 'step_size', 'num_steps', 'normalize',
                         'refresh_every', 'test_accuracy', 'delta_accuracy',
                         'relative_displacement', 'best_epoch']])
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.barplot(data=guidance_df, x='variant', y='delta_accuracy', hue='dataset', ax=ax)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title('Strategy 2 guidance knobs — ΔAcc vs the linear probe (subset seed 0)')
    plt.xticks(rotation=30, ha='right'); plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / 'sweep_guidance.png', dpi=150); plt.show()
else:
    print('Guidance sweep disabled (RUN_GUIDANCE_SWEEP = False).')

## 16. Optional extension — jointly fine-tuning the classifier

After the frozen-classifier experiments, the head is unfrozen and optimized jointly with the FM, at
its own smaller learning rate.

**This comparison needs a control, or it is uninterpretable.** Unfreezing the head adds two things
at once: the FM *and* additional training of the classifier itself. A joint run that beats the
Stage 1 probe may simply be a probe that trained longer. So a `head_only` control is run alongside:
the same Stage 1 head, continued for the same epoch budget under the same optimizer and the same
validation-checkpointing rule, with **no FM at all**. The number that means anything is joint versus
that control, not joint versus Stage 1.

In [ ]:
RUN_JOINT_FINETUNE = True

def train_head_only(dataset_name, subset_seed, run_dir):
    """Control for the joint run: continue training the Stage 1 head alone, same budget, same rule,
    no FM. Isolates 'more head training' from 'the FM'."""
    probe = PROBES[(dataset_name, subset_seed)]
    z_tr, y_tr, z_va, y_va, z_te, y_te = STAGE3_DATA[(dataset_name, subset_seed)]
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(INIT_SEED)
    head = copy.deepcopy(probe['head']); head.requires_grad_(True); head.train()
    optimizer = torch.optim.AdamW(head.parameters(), lr=S3['head_learning_rate'],
                                  weight_decay=S3['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                                                            patience=S3['lr_patience'])
    best_acc, best_state, best_epoch, stale, history = -1.0, None, None, 0, []
    init_acc, init_loss, _ = evaluate_system(None, head, z_va, y_va, T)
    best_acc, best_epoch = init_acc, 0
    best_state = copy.deepcopy(head.state_dict())
    history.append(dict(epoch=0, train_loss=float('nan'), val_accuracy=init_acc, val_loss=init_loss))
    n_train = z_tr.shape[0]
    started = time.perf_counter()
    for epoch in range(1, S3['max_epochs'] + 1):
        generator = torch.Generator().manual_seed(INIT_SEED * 100000 + epoch)
        perm = torch.randperm(n_train, generator=generator)
        head.train(); total = 0.0
        for start in range(0, n_train, S3['batch_size']):
            idx = perm[start:start + S3['batch_size']]
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(head(z_tr[idx]), y_tr[idx])
            loss.backward(); optimizer.step(); total += loss.item() * len(idx)
        val_acc, val_loss, _ = evaluate_system(None, head, z_va, y_va, T)
        history.append(dict(epoch=epoch, train_loss=total / n_train, val_accuracy=val_acc, val_loss=val_loss))
        if val_acc > best_acc + S3['min_delta']:
            best_acc, best_epoch = val_acc, epoch
            best_state = copy.deepcopy(head.state_dict()); stale = 0
        else:
            stale += 1
        scheduler.step(val_acc)
        if stale >= S3['early_stopping_patience']: break
    head.load_state_dict(best_state); head.eval()
    test_acc, test_ce, test_pred = evaluate_system(None, head, z_te, y_te, T)
    metrics = dict(dataset=dataset_name, encoder=ENCODER, shot=str(SHOT), subset_seed=subset_seed,
                   strategy='head_only', tag='head_only', T=None, joint_finetune=True,
                   baseline_test_accuracy=probe['stage1_test_accuracy'],
                   best_val_accuracy=best_acc, best_epoch=best_epoch, epochs_trained=len(history) - 1,
                   test_accuracy=test_acc, test_loss=test_ce,
                   delta_accuracy=test_acc - probe['stage1_test_accuracy'],
                   mean_displacement=0.0, relative_displacement=0.0,
                   runtime_seconds=time.perf_counter() - started,
                   feature_transform=probe['feature_transform'])
    (run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    np.save(run_dir / 'test_predictions.npy', test_pred.numpy())
    return metrics, history, test_pred

if RUN_JOINT_FINETUNE:
    joint_rows = []
    for dataset_name in DATASETS:
        print(f'\n=== joint fine-tuning | {dataset_name} ===')
        for subset_seed in SUBSET_SEEDS:
            probe = PROBES[(dataset_name, subset_seed)]
            joint_rows.append(dict(
                dataset=dataset_name, strategy='linear_probe', subset_seed=subset_seed,
                test_accuracy=probe['stage1_test_accuracy'], delta_accuracy=0.0,
                best_epoch=None, relative_displacement=0.0))
            control_dir = run_directory('head_only', dataset_name, subset_seed, OUTPUT_ROOT / 'extension')
            control_metrics_path = control_dir / 'metrics.json'
            if control_metrics_path.exists() and not FORCE_RETRAIN:
                control = json.loads(control_metrics_path.read_text())
                print(f"  loaded  head_only                {dataset_name:11s} subset {subset_seed}  "
                      f"test {control['test_accuracy']:.4f}")
            else:
                control, _, _ = train_head_only(dataset_name, subset_seed, control_dir)
                print(f"  trained head_only                {dataset_name:11s} subset {subset_seed}  "
                      f"test {control['test_accuracy']:.4f} ({control['delta_accuracy']:+.4f})")
            joint_rows.append(control)
            for strategy, _label in STRATEGIES:
                metrics, _, _ = run_one(strategy, dataset_name, subset_seed,
                                        tag=f'{strategy}__joint', joint=True,
                                        root=OUTPUT_ROOT / 'extension')
                joint_rows.append(dict(metrics, strategy=f'{strategy}__joint'))
    joint_df = pd.DataFrame(joint_rows)
    joint_df.to_csv(OUTPUT_ROOT / 'joint_finetune.csv', index=False)
    joint_summary = (joint_df.groupby(['dataset', 'strategy'], as_index=False)
                     .agg(mean_accuracy=('test_accuracy', 'mean'), std_accuracy=('test_accuracy', 'std'),
                          mean_delta=('delta_accuracy', 'mean')))
    display(joint_summary)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=joint_summary, x='dataset', y='mean_accuracy', hue='strategy', ax=ax, order=DATASETS)
    ax.set_title(f'Optional extension — unfrozen classifier vs the head-only control ({ENCODER}, K={SHOT})')
    ax.legend(fontsize=8); plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / 'joint_finetune.png', dpi=150); plt.show()
    print('\nRead joint__* against head_only, not against linear_probe: the head-only column is what '
          '"the same amount of extra classifier training, without any FM" achieves.')
else:
    print('Joint fine-tuning extension disabled (RUN_JOINT_FINETUNE = False).')

## 17. Summary

Everything the specification asks Stage 3 to report, in one place. The numbers below are read
straight from the saved artifacts; nothing is recomputed here.

In [ ]:
print(f'Stage 3 — FM before a frozen linear classifier')
print(f'  encoder {ENCODER} | K={SHOT} | T={T} | subset seeds {SUBSET_SEEDS}')
print(f'  {len(results_df[results_df.strategy != "linear_probe"])} trained FM layers, '
      f'{len(results_df)} result rows in the main grid\n')

final = summary.pivot(index='dataset', columns='strategy', values='mean_accuracy').reindex(DATASETS)
final_delta = summary.pivot(index='dataset', columns='strategy', values='mean_delta').reindex(DATASETS)
report = pd.DataFrame({
    'linear probe': final['linear_probe'],
    'end-to-end rollout': final['e2e'],
    'Δ e2e': final_delta['e2e'],
    'classifier-guided': final['guided'],
    'Δ guided': final_delta['guided'],
}).round(4)
display(report)
report.to_csv(OUTPUT_ROOT / 'stage3_report_table.csv')

wins = significance_df.groupby('strategy').agg(
    cells=('delta', 'size'), positive=('delta', lambda s: int((s > 0).sum())),
    negative=('delta', lambda s: int((s < 0).sum())),
    ci_excludes_zero=('ci_excludes_zero', 'sum'), mcnemar_significant=('significant', 'sum'))
display(wins)
print('\nArtifacts written to:', OUTPUT_ROOT)
for name in sorted(p.name for p in OUTPUT_ROOT.glob('*.csv')): print('  ', name)
for name in sorted(p.name for p in OUTPUT_ROOT.glob('*.png')): print('  ', name)